# Topic modeling (LDA) — exploratory pass

Explores what policy topics are discussed in state-parliament speeches, scoped to each state's
legislative period immediately before vs. after AfD entry. `CORPUS` switches the unit: one document per
speech (the pre/post-AfD periods), or one per agenda item (TOP) from 2009-09-29 on, built from the
TOP-opener classifier's output (opener heading + the item's speeches). This does not measure morality or
politeness — the topic assigned to each speech is meant as a **control** variable for later
analysis (some topics may be more polarized/moralized than others, independent of AfD entry).

Runs two independent topic-count-selection methods and compares them: gensim LDA scored by c_v
coherence, and sklearn LDA scored by held-out log-likelihood (the latter per the practical guide
https://medium.com/data-science/practical-guide-to-topic-modeling-with-lda-05cd6b027bdf, which
argues coherence is unreliable for tuning -- rather than pick a side, both run and get compared).

See `docs/superpowers/specs/2026-08-14-topic-modeling-lda-design.md` (local-only, not tracked in
git) for full design rationale.

In [1]:
import os
import sys

import spacy
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath(".."))
load_dotenv("../.env")

from topic_modeling_lib import (
    build_gensim_corpus,
    build_sklearn_corpus,
    gensim_coherence_scan,
    load_cache,
    load_corpus,
    make_spacy_preprocessor,
    save_cache,
    sklearn_loglikelihood_search,
    sklearn_topic_words,
    timed,
    tokenize_by_paragraph,
    tokenize_in_chunks,
)

DATA_ROOT = os.environ["DATA_ROOT"]
timing_log = []

## Parameters

Start small (single state, sampled) to get a fast timing read before scaling up -- see the
"Timing" cell at the end for what that read implies about a full run.

In [2]:
CORPUS = "tops_since_2009-09-29.parquet"   # one document per agenda item (TOP);
                                           # "speeches_afd_prepost.parquet" = one per speech
MIN_WORDS = 50          # drop shorter documents; None = keep all
STATES = None       # None = all 16 states
PRE_POST = None        # "pre", "post", or None for both (speech corpus only)
SAMPLE_N = None         # None = use every document in scope
SAMPLE_SEED = 20260814  # random_state for load_corpus's sampling
MODEL_SEED = 20261003   # random_state for the LDA fits
K_RANGE = [10, 20, 30, 40, 50, 60]
SKLEARN_N_ITER = None   # None = try every k in K_RANGE
N_PROCESS = 2           # spaCy worker processes; same tokens, faster
BATCH_SIZE = 200        # texts per spaCy batch
BY_PARAGRAPH = True     # tokenize paragraph by paragraph and join per document; bounds memory
                        # by the longest paragraph (TOPs run to 100k words)
CHUNK_SIZE = 2000       # documents per cached preprocessing chunk (an interrupted run resumes)

# Cache key inputs for everything downstream. SAMPLE_SEED is in here because a different
# random sample of the same size is a different corpus, not a cache hit. "preprocessing" is
# filled in from the actual loaded spaCy model in the next cell, so it can never drift from
# the model that produced the tokens.
run_params = {
    "corpus": CORPUS,
    "min_words": MIN_WORDS,
    "by_paragraph": BY_PARAGRAPH,
    "states": STATES,
    "pre_post": PRE_POST,
    "sample_n": SAMPLE_N,
    "sample_seed": SAMPLE_SEED,
}

In [3]:
with timed("load_corpus", log=timing_log):
    corpus_df = load_corpus(
        DATA_ROOT, states=STATES, pre_post=PRE_POST, sample_n=SAMPLE_N, seed=SAMPLE_SEED,
        corpus_file=CORPUS, min_words=MIN_WORDS,
    )
print(f"{len(corpus_df):,} documents loaded")

with timed("load spaCy model", log=timing_log):
    nlp = spacy.load("de_core_news_lg")
run_params["preprocessing"] = f"{nlp.meta['lang']}_{nlp.meta['name']}_{nlp.meta['version']}"
print(f"preprocessing: {run_params['preprocessing']}")
preprocess = make_spacy_preprocessor(nlp, n_process=N_PROCESS, batch_size=BATCH_SIZE)

# Cached in chunks, separately from the K-scans below: this doesn't depend on K_RANGE at all,
# only on which documents were loaded and which spaCy model tokenized them.
with timed("preprocess", log=timing_log):
    tokenize = (lambda chunk: tokenize_by_paragraph(preprocess, chunk)) if BY_PARAGRAPH else preprocess
    tokenized_docs = tokenize_in_chunks(tokenize, corpus_df["text"].tolist(), run_params, chunk_size=CHUNK_SIZE)
print(f"Example tokens: {tokenized_docs[0][:10]}")

[load_corpus] 23.24s
73,599 documents loaded


[load spaCy model] 0.51s
preprocessing: de_core_news_lg_3.8.0


[chunk 1/37] 2000 documents in 333s


[chunk 2/37] 2000 documents in 278s


[chunk 3/37] 2000 documents in 163s


...

[chunk 35/37] 2000 documents in 295s


[chunk 36/37] 2000 documents in 284s


[chunk 37/37] 1599 documents in 190s


[preprocess] 11106.69s


Example tokens: ['danken', 'nunmehr', 'beschlussfassung', 'vorläufig', 'geschäftsordnung', 'antrag', 'abgeordneter', 'heppener', 'drucksache', 'gemäß']


## Vectorize and fit both models across K

Both methods share the same `tokenized_docs` and `K_RANGE`, but vectorize independently (gensim's
`Dictionary`/BoW vs. sklearn's `CountVectorizer`/doc-term matrix) since each library needs its own
input format. Results are cached under `measurement/run_history/topic_modeling/`, keyed by
`run_params` + K range + method + model seed -- rerunning this notebook with the same parameters
loads from disk instead of re-fitting.

In [4]:
with timed("build_gensim_corpus", log=timing_log):
    dictionary, gensim_corpus = build_gensim_corpus(tokenized_docs)

with timed("gensim_coherence_scan (all k)", log=timing_log):
    gensim_results = gensim_coherence_scan(
        tokenized_docs, dictionary, gensim_corpus, k_range=K_RANGE, params=run_params, seed=MODEL_SEED,
    )
gensim_results[["k", "coherence", "seconds"]]

[build_gensim_corpus] 57.20s


[gensim k=10] 185.77s, coherence=0.3552


[gensim k=20] 274.89s, coherence=0.3943


[gensim k=30] 341.99s, coherence=0.4125


[gensim k=40] 399.91s, coherence=0.4068


[gensim k=50] 507.04s, coherence=0.4355


[gensim k=60] 540.24s, coherence=0.4590


[gensim_coherence_scan (all k)] 2252.22s


,k,coherence,seconds
0,10,0.355210,185.769281
1,20,0.394310,274.890340
2,30,0.412463,341.994087
3,40,0.406751,399.909508
4,50,0.435508,507.040301
5,60,0.458971,540.242699


In [5]:
with timed("build_sklearn_corpus", log=timing_log):
    vectorizer, dtm = build_sklearn_corpus(tokenized_docs)

with timed("sklearn_loglikelihood_search (all k)", log=timing_log):
    sklearn_results = sklearn_loglikelihood_search(
        dtm, vectorizer, tokenized_docs, dictionary,
        k_range=K_RANGE, params=run_params, n_iter=SKLEARN_N_ITER, seed=MODEL_SEED,
    )
sklearn_results[["k", "log_likelihood", "coherence", "seconds"]]

[build_sklearn_corpus] 18.69s


[sklearn k=40] 830.78s, log_likelihood=-217616134.26, coherence=0.5030


[sklearn k=60] 997.14s, log_likelihood=-221110702.62, coherence=0.5037


[sklearn k=10] 564.39s, log_likelihood=-213076118.71, coherence=0.4404


[sklearn k=30] 702.12s, log_likelihood=-215950028.05, coherence=0.4778


[sklearn k=50] 905.17s, log_likelihood=-219330157.33, coherence=0.5093


[sklearn k=20] 651.10s, log_likelihood=-214577041.91, coherence=0.4470


[sklearn_loglikelihood_search (all k)] 4652.15s


,k,log_likelihood,coherence,seconds
0,40,-2.176161e+08,0.503009,830.783510
1,60,-2.211107e+08,0.503674,997.138686
2,10,-2.130761e+08,0.440396,564.392099
3,30,-2.159500e+08,0.477808,702.117710
4,50,-2.193302e+08,0.509324,905.169310
5,20,-2.145770e+08,0.447015,651.095395


## Compare the two methods' preferred K

In [6]:
best_gensim_k = int(gensim_results.loc[gensim_results["coherence"].idxmax(), "k"])
best_sklearn_k_coherence = int(sklearn_results.loc[sklearn_results["coherence"].idxmax(), "k"])
best_sklearn_k_loglik = int(sklearn_results.loc[sklearn_results["log_likelihood"].idxmax(), "k"])

print(f"gensim (c_v coherence) prefers k={best_gensim_k}")
print(f"sklearn (c_v coherence) prefers k={best_sklearn_k_coherence}")
print(f"sklearn (log-likelihood) prefers k={best_sklearn_k_loglik}")
print()
print("Same-metric comparison (coherence, both libraries):",
      "Agreement" if best_gensim_k == best_sklearn_k_coherence
      else "Disagreement -- the two implementations of the same measure land on different K")
print("Cross-metric comparison (gensim coherence vs. sklearn log-likelihood):",
      "Agreement" if best_gensim_k == best_sklearn_k_loglik
      else "Disagreement -- inspect both before picking K")

gensim (c_v coherence) prefers k=60
sklearn (c_v coherence) prefers k=50
sklearn (log-likelihood) prefers k=10

Same-metric comparison (coherence, both libraries): Disagreement -- the two implementations of the same measure land on different K
Cross-metric comparison (gensim coherence vs. sklearn log-likelihood): Disagreement -- inspect both before picking K


## Inspect topics for the chosen K

In [7]:
chosen_k = best_gensim_k  # change after inspecting the comparison above

print(f"=== gensim topics (k={chosen_k}) ===\n")
chosen_gensim_model = gensim_results.set_index("k").loc[chosen_k, "model"]
for topic_id, terms in chosen_gensim_model.print_topics(num_topics=-1, num_words=10):
    print(f"Topic {topic_id}: {terms}\n")

sklearn_chosen_k = best_sklearn_k_coherence
print(f"\n=== sklearn topics (k={sklearn_chosen_k}, its own coherence-preferred K) ===\n")
chosen_sklearn_model = sklearn_results.set_index("k").loc[sklearn_chosen_k, "model"]
for topic_id, words in enumerate(sklearn_topic_words(chosen_sklearn_model, vectorizer, topn=10)):
    print(f"Topic {topic_id}: {', '.join(words)}\n")

=== gensim topics (k=60) ===



Topic 0: 0.019*"justiz" + 0.009*"herr" + 0.008*"staatsanwaltschaft" + 0.008*"gericht" + 0.007*"richter" + 0.007*"opfer" + 0.006*"fall" + 0.006*"dame" + 0.005*"rechtsstaat" + 0.005*"frau"

Topic 1: 0.048*"kind" + 0.030*"jugendliche" + 0.012*"jung" + 0.009*"schule" + 0.008*"antrag" + 0.008*"inklusion" + 0.007*"eltern" + 0.007*"wichtig" + 0.007*"herr" + 0.006*"bildung"

Topic 2: 0.055*"öpnv" + 0.026*"mobilität" + 0.024*"fahren" + 0.021*"bus" + 0.015*"auto" + 0.015*"ticket" + 0.014*"öffentlich" + 0.013*"angebot" + 0.013*"nahverkehr" + 0.011*"aufgabenträger"

Topic 3: 0.082*"verbraucherschutz" + 0.054*"verbraucher" + 0.036*"verbraucherzentrale" + 0.026*"versicherung" + 0.025*"verbraucherin" + 0.016*"kunde" + 0.014*"versorgungswerk" + 0.012*"konto" + 0.010*"sparkasse" + 0.009*"schuldner"

Topic 4: 0.019*"land" + 0.019*"haushalt" + 0.015*"million" + 0.013*"herr" + 0.010*"euro" + 0.009*"milliarde" + 0.007*"sagen" + 0.007*"investition" + 0.006*"dame" + 0.006*"schuld"

Topic 5: 0.084*"kommune" +

Topic 0: sprache, mensch, diskriminierung, frau, sexuell, deutsch, antrag, gesellschaft, vielfalt, ehe

Topic 1: sachsen, sächsisch, staatsregierung, freistaat, herr, geehrt, frei, antrag, dresden, leipzig

Topic 2: herr, dame, frage, fall, sagen, stellen, frau, wissen, kollege, verfassungsschutz

Topic 3: wald, fläche, naturschutz, klimawandel, wasser, natur, klimaschutz, baum, nationalpark, hektar

Topic 4: hessen, herr, hessisch, landesregierung, dame, flughafen, sagen, grünen, frage, druck

Topic 5: herr, sagen, land, dame, kollege, straße, bund, öpnv, bahn, fahren

Topic 6: polizei, herr, sicherheit, kollege, feuerwehr, einsatz, polizist, dame, sagen, kollegin

Topic 7: mitglied, wahl, fraktion, abgeordneter, herr, stimme, tagesordnungspunkt, wählen, cdu, wahlvorschlag

Topic 8: herr, land, kollege, dame, rundfunk, sagen, wichtig, digital, ministerpräsident, thema

Topic 9: kommune, kommunal, herr, land, gemeinde, euro, mal, dame, stadt, sagen

Topic 10: antrag, fraktion, drucksac

## Timing summary — for estimating full-corpus runtime

This ran on the sample size and state(s) set in the Parameters cell above. Multiply the
preprocess/vectorize/fit seconds-per-document by the full pre/post-AfD corpus size (see
`preprocessing/afd_period_window.py`'s printed document count) to estimate a full run's cost
before committing to it.

In [8]:
import pandas as pd

timing_df = pd.DataFrame(timing_log)
timing_df["seconds_per_doc"] = timing_df["seconds"] / len(corpus_df)
timing_df

,step,seconds,seconds_per_doc
0,load_corpus,23.244283,0.000316
1,load spaCy model,0.509558,0.000007
2,preprocess,11106.689612,0.150908
3,build_gensim_corpus,57.195041,0.000777
4,gensim_coherence_scan (all k),2252.217827,0.030601
5,build_sklearn_corpus,18.687434,0.000254
6,sklearn_loglikelihood_search (all k),4652.151154,0.063209
